1. PassengerID : 탑승객 고유 아이디
2. Survival : 탑승객 생존 유무 (0: 사망, 1: 생존)
3. Pclass : 등실의 등급
4. Name : 이름
5. Sex : 성별
6. Age : 나이
7. Sibsp : 함께 탐승한 형제자매, 아내, 남편의 수
8. Parch : 함께 탐승한 부모, 자식의 수
9. Ticket :티켓 번호
10. Fare : 티켓의 요금
11. Cabin : 객실번호
12. Embarked : 배에 탑승한 항구 이름 ( C = Cherbourn, Q = Queenstown, S = Southampton)

## 라이브러리 가져오기

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## 데이터 가져오기

In [2]:
train = pd.read_csv('1.titanic_train.csv')
test = pd.read_csv('2.titanic_test.csv')
submission = pd.read_csv('3.titanic_submission.csv')

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.5000,A6,S
1,339,1,3,"Dahl, Mr. Karl Edwart",male,45.0,0,0,7598,8.0500,NaN,S
2,769,0,3,"Moran, Mr. Daniel J",male,NaN,1,0,371110,24.1500,NaN,Q
3,692,1,3,"Karun, Miss. Manca",female,4.0,0,1,349256,13.4167,NaN,C
4,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.7500,NaN,Q


## 전처리에 사용하지 않을 컬럼은 제거

In [3]:
train = train.drop(['PassengerId', 'Ticket', 'Cabin', 'Name'], axis=1)
test = test.drop(['PassengerId', 'Ticket', 'Cabin', 'Name'], axis=1)
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,1,male,28.0,0,0,35.5000,S
1,1,3,male,45.0,0,0,8.0500,S
2,0,3,male,NaN,1,0,24.1500,Q
3,1,3,female,4.0,0,1,13.4167,C
4,0,3,male,32.0,0,0,7.7500,Q


## 결측치 처리

In [4]:
train.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         146
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [5]:
train.Embarked.value_counts()

Embarked
S    538
C    153
Q     64
Name: count, dtype: int64

위 내용을 기준으로 Embarked는 가장 많은 비중을 차지 하는 것이 S이므로 Embarked의 빈값은 S로 채우기로 한다

In [6]:
train['Embarked'] = train.Embarked.fillna('S')
train.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         146
SibSp         0
Parch         0
Fare          0
Embarked      0
dtype: int64

### 나이는 쉽게 전부 평균으로 취급하는 것으로

In [7]:
train['Age'] = train.Age.fillna(train.Age.mean())
train

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,1,male,28.000000,0,0,35.5000,S
1,1,3,male,45.000000,0,0,8.0500,S
2,0,3,male,29.646481,1,0,24.1500,Q
3,1,3,female,4.000000,0,1,13.4167,C
4,0,3,male,32.000000,0,0,7.7500,Q
...,...,...,...,...,...,...,...,...
752,0,3,female,29.646481,0,2,15.2458,C
753,0,3,male,17.000000,0,0,8.6625,S
754,0,3,female,6.000000,4,2,31.2750,S
755,0,3,male,21.000000,0,0,7.7333,Q


In [8]:
train.isna().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

## 성별은 1(남자), 0(여자)로 구분

In [9]:
train['Sex'] = train.Sex.apply(lambda x: 1 if x == 'male' else 0)
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,1,1,28.000000,0,0,35.5000,S
1,1,3,1,45.000000,0,0,8.0500,S
2,0,3,1,29.646481,1,0,24.1500,Q
3,1,3,0,4.000000,0,1,13.4167,C
4,0,3,1,32.000000,0,0,7.7500,Q


## pclass와 도착지에 대해 원핫인코딩

In [10]:
# 현재 pclass와 embarked는 바로 원핫인코딩 적용
embarked = pd.get_dummies(train.Embarked)
pclass = pd.get_dummies(train.Pclass)

In [11]:
# 컬럼명을 만들기 위해 설정
pclass.columns = [f'pclass_{i}' for i in range(1,4)]
pclass.head(3)

,pclass_1,pclass_2,pclass_3
0,True,False,False
1,False,False,True
2,False,False,True


In [12]:
train_df = train.drop(['Pclass','Embarked'], axis=1)

In [13]:
train_df = pd.concat([train_df, pclass, embarked], axis=1) # 컬럼 병합
train_df.head()

,Survived,Sex,Age,SibSp,Parch,Fare,pclass_1,pclass_2,pclass_3,C,Q,S
0,1,1,28.000000,0,0,35.5000,True,False,False,False,False,True
1,1,1,45.000000,0,0,8.0500,False,False,True,False,False,True
2,0,1,29.646481,1,0,24.1500,False,False,True,False,True,False
3,1,0,4.000000,0,1,13.4167,False,False,True,True,False,False
4,0,1,32.000000,0,0,7.7500,False,False,True,False,True,False


## 가족 구성원 합치기

In [14]:
train_df['family'] = train_df.SibSp + train_df.Parch + 1
train_df.head()

,Survived,Sex,Age,SibSp,Parch,Fare,pclass_1,pclass_2,pclass_3,C,Q,S,family
0,1,1,28.000000,0,0,35.5000,True,False,False,False,False,True,1
1,1,1,45.000000,0,0,8.0500,False,False,True,False,False,True,1
2,0,1,29.646481,1,0,24.1500,False,False,True,False,True,False,2
3,1,0,4.000000,0,1,13.4167,False,False,True,True,False,False,2
4,0,1,32.000000,0,0,7.7500,False,False,True,False,True,False,1


In [15]:
train_df = train_df.drop(['SibSp', 'Parch'], axis=1)
train_df.head()

,Survived,Sex,Age,Fare,pclass_1,pclass_2,pclass_3,C,Q,S,family
0,1,1,28.000000,35.5000,True,False,False,False,False,True,1
1,1,1,45.000000,8.0500,False,False,True,False,False,True,1
2,0,1,29.646481,24.1500,False,False,True,False,True,False,2
3,1,0,4.000000,13.4167,False,False,True,True,False,False,2
4,0,1,32.000000,7.7500,False,False,True,False,True,False,1


In [ ]:
# 가족수만큼 나눠서 비용 맞추기
train_df['Fare'] = train_df.Fare/train_df.family
train_df.head()

,Survived,Sex,Age,Fare,pclass_1,pclass_2,pclass_3,C,Q,S,family
0,1,1,28.000000,35.50000,True,False,False,False,False,True,1
1,1,1,45.000000,8.05000,False,False,True,False,False,True,1
2,0,1,29.646481,12.07500,False,False,True,False,True,False,2
3,1,0,4.000000,6.70835,False,False,True,True,False,False,2
4,0,1,32.000000,7.75000,False,False,True,False,True,False,1


## test 데이터셋 전처리

In [19]:
# 1차 데이터 정리
test.isna().sum()

Pclass       0
Sex          0
Age         31
SibSp        0
Parch        0
Fare         0
Embarked     0
dtype: int64

In [20]:
# 데이터 메꾸기
test['Age'] = test.Age.fillna(test.Age.mean())
test.isna().sum()

Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [21]:
test.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,2,female,18.000000,0,2,13.00,S
1,1,female,17.000000,1,0,108.90,C
2,3,male,30.011359,0,0,8.05,S
3,1,female,39.000000,1,0,55.90,S
4,2,male,60.000000,1,1,39.00,S


In [22]:
# 성별 1, 0으로 분류
test['Sex'] = train.Sex.apply(lambda x: 1 if x == 'male' else 0)

In [23]:
# 원핫인코딩 및 병합
test_embarked = pd.get_dummies(test.Embarked)
test_pclass = pd.get_dummies(test.Pclass)
test_pclass.columns = [f'class_{i}' for i in range(1,4)]
test_df = pd.concat([test, test_pclass, test_embarked], axis=1)
test_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,class_1,class_2,class_3,C,Q,S
0,2,0,18.000000,0,2,13.00,S,False,True,False,False,False,True
1,1,0,17.000000,1,0,108.90,C,True,False,False,True,False,False
2,3,0,30.011359,0,0,8.05,S,False,False,True,False,False,True
3,1,0,39.000000,1,0,55.90,S,True,False,False,False,False,True
4,2,0,60.000000,1,1,39.00,S,False,True,False,False,False,True


In [24]:
# 원핫인코딩을 했으므로 embarked, pclass 컬럼 삭제
test_df = test_df.drop(['Pclass','Embarked'], axis=1)
test_df.head()

,Sex,Age,SibSp,Parch,Fare,class_1,class_2,class_3,C,Q,S
0,0,18.000000,0,2,13.00,False,True,False,False,False,True
1,0,17.000000,1,0,108.90,True,False,False,True,False,False
2,0,30.011359,0,0,8.05,False,False,True,False,False,True
3,0,39.000000,1,0,55.90,True,False,False,False,False,True
4,0,60.000000,1,1,39.00,False,True,False,False,False,True


In [25]:
# 가족으로 합치기
test_df['family'] = test_df.SibSp + test_df.Parch + 1
test_df.drop(['SibSp', 'Parch'], axis=1, inplace=True)
test_df.head()

,Sex,Age,Fare,class_1,class_2,class_3,C,Q,S,family
0,0,18.000000,13.00,False,True,False,False,False,True,3
1,0,17.000000,108.90,True,False,False,True,False,False,2
2,0,30.011359,8.05,False,False,True,False,False,True,1
3,0,39.000000,55.90,True,False,False,False,False,True,2
4,0,60.000000,39.00,False,True,False,False,False,True,3


In [26]:
test_df['Fare'] = test_df.Fare/test_df.family
test_df.head()

,Sex,Age,Fare,class_1,class_2,class_3,C,Q,S,family
0,0,18.000000,4.333333,False,True,False,False,False,True,3
1,0,17.000000,54.450000,True,False,False,True,False,False,2
2,0,30.011359,8.050000,False,False,True,False,False,True,1
3,0,39.000000,27.950000,True,False,False,False,False,True,2
4,0,60.000000,13.000000,False,True,False,False,False,True,3


### 학습, 타겟 분리

In [27]:
target = train_df.Survived
train_df = train_df.drop('Survived', axis=1)

In [28]:
X_train, X_test, y_train, y_test = train_test_split(train_df, target, random_state=42) # 기본 testset 0.25

# 모델 학습 및 평가

In [29]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier(random_state=42)
params = {
    'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
    'max_depth': range(4,20),
    'min_samples_split': range(2,100,10)
}
gs = GridSearchCV(rf_clf, param_grid=params, n_jobs=-1)
gs.fit(X_train, y_train)
rf = gs.best_estimator_ # <-- RandomForestClassifier
print(rf.score(X_train, y_train))
p = rf.predict(X_test)
accuracy_score(p, y_test)

0.855379188712522


0.7842105263157895

In [31]:
from xgboost import XGBClassifier
clf = XGBClassifier()
clf.fit(X_train, y_train)
print(clf.score(X_train, y_train))
accuracy_score(clf.predict(X_test), y_test)

0.9805996472663139


0.7947368421052632

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
dt_clf = DecisionTreeClassifier(random_state=42)
params = {
    'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
    'max_depth': range(4,15),
    'min_samples_split': range(2,100,10)
}
gs = GridSearchCV(dt_clf, param_grid=params, n_jobs=-1)
gs.fit(X_train, y_train)
dt = gs.best_estimator_ # <-- DecisionTreeClass
print(dt.score(X_train, y_train))
p = dt.predict(X_test)
accuracy_score(p, y_test)

0.8271604938271605


0.7578947368421053

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
svm = SVC(kernel='linear',random_state=42)
svm.fit(X_train, y_train)
print(svm.score(X_train, y_train))
p = svm.predict(X_test)
accuracy_score(p, y_test)

0.8112874779541446


0.7578947368421053

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(X_train, y_train)
print(knn.score(X_train, y_train))
accuracy_score(knn.predict(X_test), y_test)

0.8359788359788359


0.6684210526315789

In [ ]:
from sklearn.linear_model import LogisticRegression
lg = LogisticRegression(random_state=42)
lg.fit(X_train, y_train)
print(lg.score(X_train, y_train))
accuracy_score(lg.predict(X_test), y_test)

0.8218694885361552


c:\Users\USER\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7842105263157895

모델 성능으론 랜덤포레스트-로지스틱회귀-SVM==결정트리-KNN 순임

# 제출 데이터셋

In [ ]:
submission['Survived'] = pred#.astype(bool)
submission

,PassengerId,Survived
0,418,1
1,308,1
2,88,0
3,578,1
4,685,0
...,...,...
129,223,0
130,859,1
131,475,0
132,347,1


In [ ]:
submission.to_csv('20250721_test.csv', encoding='utf-8', index=False)